# 07 — Contrato com contexto

**Trilha:** [06 roteamento por enum](../02_intermediario/06_roteamento_por_enum.ipynb) → **07 contrato e contexto** → [08 capacidades e políticas](08_capacidades_e_politicas.ipynb) → [09 runtime](09_runtime_com_dataclasses.ipynb) → [10 pipeline](10_pipeline_completo.ipynb)

**De onde viemos:** no [05](../02_intermediario/05_regras_e_retry.ipynb), as regras de negócio eram funções **fora** do
contrato — dava para esquecer de chamá-las — e não sabiam nada do pedido (orçamento, despensa, preços). Aqui as regras entram
**no contrato** e recebem esse contexto pelo `context=` do Pydantic. Quem valida a lista, valida as regras.

A partir daqui o modelo é **simulado** ([`llm_simulado.py`](llm_simulado.py)): um cliente com a mesma interface do `OpenAI(...)`
(`.chat.completions.create`) que devolve objetos do próprio SDK, seguindo um roteiro. Tudo roda e pode ser testado sem rede.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from logger_general import GeneralLogger
from prettifier import show, show_table

In [2]:
import json
from dataclasses import dataclass, field
from decimal import ROUND_HALF_UP, Decimal
from types import MappingProxyType
from typing import Mapping

from pydantic import BaseModel, PrivateAttr, ValidationError, ValidationInfo, model_validator

from contrato import ListaDeCompras, Unidade, extrair_json, prompt_com_schema
from llm_simulado import ClienteSimulado

## 1. Unidades e dinheiro

Para comparar com a despensa e calcular custo, as quantidades precisam ser convertidas entre unidades — e o dinheiro passa a ser
`Decimal` (o `float` do [04](../02_intermediario/04_ferramentas_com_funcoes.ipynb) acumulava erro).

In [3]:
FATOR = {
    (Unidade.G, Unidade.KG): Decimal("0.001"), (Unidade.KG, Unidade.G): Decimal(1000),
    (Unidade.ML, Unidade.L): Decimal("0.001"), (Unidade.L, Unidade.ML): Decimal(1000),
}


def converter(quantidade: Decimal, de: Unidade, para: Unidade) -> Decimal:
    if de is para:
        return quantidade
    try:
        return quantidade * FATOR[(de, para)]
    except KeyError:
        raise ValueError(f"não dá para converter {de} em {para}") from None


show(f"converter(250 g -> kg): {converter(Decimal(250), Unidade.G, Unidade.KG)}")
try:
    converter(Decimal(250), Unidade.G, Unidade.UN)
except ValueError as ex:
    show(f"ValueError: {ex}")

converter(250 g -> kg): 0.250
ValueError: não dá para converter g em un


## 2. O contexto do pedido

Tudo o que a validação precisa saber e que **não vem do modelo**: orçamento, preços, despensa e limites. Uma dataclass
congelada, com os dicionários protegidos por `MappingProxyType` (o motivo está na pegadinha 4).

In [4]:
@dataclass(frozen=True)
class ContextoPedido:
    orcamento: Decimal
    precos: Mapping[str, tuple[Decimal, Unidade]]
    despensa: Mapping[str, tuple[Decimal, Unidade]] = field(default_factory=dict)
    limite_por_pessoa: Mapping[Unidade, Decimal] = field(default_factory=lambda: {
        Unidade.G: Decimal(400), Unidade.KG: Decimal("0.4"), Unidade.ML: Decimal(300),
        Unidade.L: Decimal("0.3"), Unidade.UN: Decimal(4),
    })

    def __post_init__(self):
        for nome in ("precos", "despensa", "limite_por_pessoa"):
            object.__setattr__(self, nome, MappingProxyType(dict(getattr(self, nome))))


PRECOS = {
    "farinha de trigo": (Decimal("6.50"), Unidade.KG),
    "muçarela de búfala": (Decimal("89.90"), Unidade.KG),
    "tomate italiano": (Decimal("1.20"), Unidade.UN),
    "azeite": (Decimal("45.00"), Unidade.L),
    "manjericão fresco": (Decimal("4.00"), Unidade.UN),
}
contexto = ContextoPedido(orcamento=Decimal("40"), precos=PRECOS, despensa={"azeite": (Decimal("0.5"), Unidade.L)})
show(f"contexto.orcamento: {contexto.orcamento}")
show(f"contexto.despensa: {dict(contexto.despensa)}")

contexto.orcamento: 40
contexto.despensa: {'azeite': (Decimal('0.5'), <Unidade.L: 'l'>)}


## 3. As regras dentro do contrato

`ListaComContexto` herda o contrato do básico e acrescenta um `model_validator` que lê `info.context`. Ele confere o limite por
pessoa (a regra do 05), desconta a despensa, calcula o custo do que falta comprar e compara com o orçamento. O custo calculado
fica num atributo privado — só existe em listas que passaram pela validação.

In [5]:
class ListaComContexto(ListaDeCompras):
    _custo: Decimal | None = PrivateAttr(default=None)

    @model_validator(mode="after")
    def regras_com_contexto(self, info: ValidationInfo):
        ctx = info.context
        if not isinstance(ctx, ContextoPedido):
            raise ValueError("contexto obrigatório: valide com context=ContextoPedido(...)")
        problemas, total = [], Decimal(0)
        for item in self.ingredientes:
            quantidade = Decimal(str(item.quantidade))
            limite = ctx.limite_por_pessoa[item.unidade] * self.porcoes
            if quantidade > limite:
                problemas.append(f"{item.nome}: {quantidade} {item.unidade} passa do limite de {limite} {item.unidade}")
                continue
            falta = quantidade
            if item.nome in ctx.despensa:
                tem, unidade_casa = ctx.despensa[item.nome]
                falta = max(Decimal(0), quantidade - converter(tem, unidade_casa, item.unidade))
            if falta == 0:
                continue
            if item.nome not in ctx.precos:
                problemas.append(f"{item.nome}: sem preço na tabela")
                continue
            preco, unidade_venda = ctx.precos[item.nome]
            try:
                total += preco * converter(falta, item.unidade, unidade_venda)
            except ValueError as ex:
                problemas.append(f"{item.nome}: {ex}")
        total = total.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        if total > ctx.orcamento:
            problemas.append(f"custo R$ {total} passa do orçamento de R$ {ctx.orcamento}")
        if problemas:
            raise ValueError("; ".join(problemas))
        self._custo = total
        return self

    @property
    def custo(self) -> Decimal:
        return self._custo

Uma resposta no formato do contrato (*exemplo construído*), validada com o contexto:

In [6]:
resposta = json.dumps({"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
    {"nome": "farinha de trigo", "quantidade": 250, "unidade": "g", "categoria": "mercearia"},
    {"nome": "muçarela de búfala", "quantidade": 200, "unidade": "g", "categoria": "laticinios"},
    {"nome": "tomate italiano", "quantidade": 4, "unidade": "un", "categoria": "hortifruti"},
    {"nome": "azeite", "quantidade": 30, "unidade": "ml", "categoria": "mercearia"},
]}, ensure_ascii=False)

lista = ListaComContexto.model_validate_json(resposta, context=contexto)
show(f"lista.custo: R$ {lista.custo}")

lista.custo: R$ 24.41


O azeite não entrou no custo: há 0,5 l na despensa e a receita pede 30 ml. Com um orçamento menor, a **mesma** resposta é
recusada — a regra depende do pedido, não só da resposta:

In [7]:
apertado = ContextoPedido(orcamento=Decimal("20"), precos=PRECOS, despensa=contexto.despensa)
try:
    ListaComContexto.model_validate_json(resposta, context=apertado)
except ValidationError as ex:
    show(f"ValidationError: {ex.errors()[0]['msg']}")

ValidationError: Value error, custo R$ 24.41 passa do orçamento de R$ 20


## 4. O retry do 05, agora com o contrato completo

O laço é o mesmo do [05](../02_intermediario/05_regras_e_retry.ipynb), mas agora só existe **uma** validação: a do contrato com
contexto. O roteiro simulado erra na primeira (muçarela demais, estoura o orçamento) e corrige na segunda.

In [8]:
INSTRUCAO = "Você gera listas de compras com ingredientes in natura e sistema métrico, em quantidades realistas."
log = GeneralLogger(name="contrato", stream=sys.stdout)


def pedir_lista(cliente, prato: str, porcoes: int, ctx: ContextoPedido, max_tentativas: int = 3) -> ListaComContexto:
    mensagens = [
        {"role": "system", "content": prompt_com_schema(INSTRUCAO, ListaDeCompras)
                                      + f"\nO custo total não pode passar de R$ {ctx.orcamento}."},
        {"role": "user", "content": f"{prato} para {porcoes} pessoas"},
    ]
    for tentativa in range(1, max_tentativas + 1):
        texto = cliente.chat.completions.create(model="gpt-4o-mini", messages=mensagens).choices[0].message.content
        try:
            lista_ok = ListaComContexto.model_validate_json(extrair_json(texto), context=ctx)
            log.info(f"tentativa {tentativa}: lista válida, custo R$ {lista_ok.custo}")
            return lista_ok
        except (ValidationError, ValueError) as ex:
            erro = ex.errors()[0]["msg"] if isinstance(ex, ValidationError) else str(ex)
            log.warning(f"tentativa {tentativa}: {erro}")
            mensagens += [{"role": "assistant", "content": texto},
                          {"role": "user", "content": f"Corrija e devolva o JSON completo: {erro}"}]
    raise RuntimeError(f"sem lista válida em {max_tentativas} tentativas")


muita_mucarela = resposta.replace('"quantidade": 200, "unidade": "g"', '"quantidade": 800, "unidade": "g"')
cliente = ClienteSimulado([muita_mucarela, resposta])
final = pedir_lista(cliente, "Pizza margherita", 2, contexto)
show(f"final.custo: R$ {final.custo}")
show(f"chamadas ao modelo: {len(cliente.chamadas)}")
show(f"última mensagem enviada: {cliente.chamadas[-1]['messages'][-1]['content']}")

2026-11-12 20:27:22 | WARNING  | tentativa 1: Value error, custo R$ 78.35 passa do orçamento de R$ 40
2026-11-12 20:27:22 | INFO     | tentativa 2: lista válida, custo R$ 24.41
final.custo: R$ 24.41
chamadas ao modelo: 2
última mensagem enviada: Corrija e devolva o JSON completo: Value error, custo R$ 78.35 passa do orçamento de 
R$ 40


## ⚠️ O que ainda quebra

**1. O modelo não vê as regras do validator.** O JSON Schema de `ListaComContexto` é **idêntico** ao de `ListaDeCompras`: nada
do `model_validator` aparece. Por isso o `pedir_lista` precisa **dizer** o orçamento no prompt — e o retry existe para o resto.

In [9]:
show(f"schemas iguais: {ListaComContexto.model_json_schema()['properties'] == ListaDeCompras.model_json_schema()['properties']}")

schemas iguais: True


**2. Saída estruturada nativa (strict) muda o schema.** Ao usar `.parse(response_format=...)` (seção 2.2 do
[02](../01_basico/02_saida_estruturada.ipynb)), o SDK converte o modelo para o modo *strict*: campos com default viram
**obrigatórios** (aceitando `null`) e o default some do schema enviado. O modelo terá de mandar `"observacao": null`
explicitamente. (A função abaixo é a que o SDK usa por dentro; fica em `openai.lib._pydantic`, um caminho privado — pode mudar.)

In [10]:
from openai.lib._pydantic import to_strict_json_schema


class ComObservacao(BaseModel):
    prato: str
    observacao: str | None = None


normal = ComObservacao.model_json_schema()
strict = to_strict_json_schema(ComObservacao)
show_table(["schema", "required", "observacao"],
       [("normal", normal["required"], normal["properties"]["observacao"]),
        ("strict", strict["required"], strict["properties"]["observacao"])])

                                                                                                              
  schema   required                  observacao                                                               
 ──────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  normal   ['prato']                 {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None,       
                                     'title': 'Observacao'}                                                   
  strict   ['prato', 'observacao']   {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'title':               
                                     'Observacao'}                                                            
                                                                                                              


**3. `model_copy` e o construtor não passam pelo mesmo caminho.** O construtor **não tem como** receber contexto, então
`ListaComContexto(...)` falha sempre. E `model_copy(update=...)` não valida nada: dá para criar uma lista acima do orçamento
com o custo antigo grudado nela.

In [11]:
try:
    ListaComContexto(prato="x", porcoes=1, ingredientes=[])
except ValidationError as ex:
    show(f"construtor: {ex.errors()[0]['msg']}")

itens = [item.model_copy(update={"quantidade": item.quantidade * 10}) for item in lista.ingredientes]
copia = lista.model_copy(update={"ingredientes": itens})
show(f"cópia com 10x a quantidade: muçarela {copia.ingredientes[1].quantidade:g} g, custo ainda R$ {copia.custo}")

construtor: Value error, contexto obrigatório: valide com context=ContextoPedido(...)
cópia com 10x a quantidade: muçarela 2000 g, custo ainda R$ 24.41


Para alterar uma lista validada, **revalide**: `ListaComContexto.model_validate(copia.model_dump(), context=ctx)`.

**4. `frozen=True` não congela o que está dentro.** Sem o `MappingProxyType` do `__post_init__`, o dicionário de preços continua
mutável — e uma mudança no meio do caminho altera o resultado de validações futuras:

In [12]:
@dataclass(frozen=True)
class ContextoIngenuo:
    precos: dict


ingenuo = ContextoIngenuo(precos={"azeite": Decimal("45.00")})
ingenuo.precos["azeite"] = Decimal("0.01")  # frozen não impediu
show(f"ingenuo.precos: {ingenuo.precos}")
try:
    contexto.precos["azeite"] = (Decimal("0.01"), Unidade.L)
except TypeError as ex:
    show(f"ContextoPedido: TypeError: {ex}")

ingenuo.precos: {'azeite': Decimal('0.01')}
ContextoPedido: TypeError: 'mappingproxy' object does not support item assignment


**5. `Decimal(float)` carrega o erro do float.** Sempre construa a partir de texto (`Decimal("0.1")`) — o validador faz
`Decimal(str(item.quantidade))` por isso:

In [13]:
show(f"Decimal(0.1): {Decimal(0.1)}")
show(f"Decimal('0.1'): {Decimal('0.1')}")

Decimal(0.1): 0.1000000000000000055511151231257827021181583404541015625
Decimal('0.1'): 0.1


**O que falta:** o contrato agora sabe **o que** é uma lista válida para este pedido. Ainda não sabe **quem** está pedindo — um
visitante não deveria finalizar compra, e o modelo não deveria conseguir chamar isso sozinho.
➡️ [08 — capacidades e políticas](08_capacidades_e_politicas.ipynb)